# Introduction

# Stage 02 — Ranks and signals

**Pipeline position:** second. Reads stage 01's eligible rows; writes the row-level analysis frame
that every later stage scores.

## What this stage answers

**How do we turn four columns of numbers into a signed disagreement with the market?**

1. **Why the stored `adp_pos_rank` cannot be used** — and what is done instead.
2. **Rank construction** across a 2x2 of rank universe x population.
3. **Signal construction** — the three signed gaps, the agreement rule, the strict thresholds, the
   consensus score, and hit/miss/tie.

## The 2x2, and why both axes are carried

**Rank universes.** `A` is the production-board analogue: rank over the whole ADP-bearing population,
so a row with no Sleeper projection keeps a missing Sleeper rank but still occupies the other three
orderings — exactly as such a player does on the real Draft Board. `B` is the required sensitivity:
restrict to rows complete on all four quantities first, then re-rank inside that identical set. A
conclusion that moves between A and B is population-sensitive and must be reported as such.

**Populations.** `all_adp` is everything; `drafted_top180` caps at `adp_overall_rank <= 180`, the
repo's own draftable universe. The cap is applied **before** ranking, so the drafted board is a
re-ranked population rather than a filtered view of the full population's ranks. That is deliberate:
they are two different questions.

## The signal, in three lines

```
model_gap   = adp_rank - model_rank      positive: we rank him above his draft price
sleeper_gap = adp_rank - sleeper_rank    positive: Sleeper ranks him above his price
actual_gap  = adp_rank - actual_rank     positive: he finished above his price
```

Agreement at threshold `t` needs matching signs **and** both magnitudes strictly greater than `t`.
The consensus score carries the shared sign on the **weaker** gap, so one extreme projection cannot
manufacture an agreement alone. A hit needs `sign(actual_gap) == sign(consensus_score)` and a nonzero
`actual_gap`; an exact tie is a **miss** in the primary rate.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | `interim/eligible_rows.csv`, `00_shared_pipeline.ipynb` |
| out | `artifacts/player_season_results.csv`, `interim/stage02_adp_rank_diagnostic.json` |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** `interim/eligible_rows.csv` and the shared library
**and writes:** `artifacts/player_season_results.csv` and `interim/stage02_adp_rank_diagnostic.json`

### Explain — load the eligible rows and show why the stored `adp_pos_rank` is unusable

Two jobs in one cell: read stage 01's handoff, then settle a question that would otherwise sit
unaddressed behind every rank in the study.

The season dataset ships an `adp_pos_rank` column. Using it directly would be convenient. This cell
shows why it would be wrong, by reconstructing a positional ADP rank two ways and comparing each
against the stored column:

1. **within the model's walk-forward population** — the rows this study evaluates;
2. **within the full ADP-bearing season dataset** — a strictly larger universe.

**What we should see, and why.** Neither reconstruction reproduces the stored column, because
`build_season_dataset.py` *merges it in verbatim* from an external ADP source where it was ranked over
that source's own universe — one containing players the season dataset never retains, and many more
the projection models never scored. A rank computed over a superset cannot be reproduced from a
subset, and widening the universe from the model population to the whole dataset should barely help.

The cell also prints a worked example: a stretch of one season-position where the stored rank skips a
value, which is the visible fingerprint of a player who exists in the ADP source but not here.

**The consequence is a real design decision.** Every rank in this study is rebuilt inside the stated
population — reproducible from the exported CSVs, at the cost of not being the literal market-wide
positional rank, and **population-dependent by construction**.

In [2]:
ELIGIBLE = pd.read_csv(INTERIM / "eligible_rows.csv")
print(f"loaded interim/eligible_rows.csv: {len(ELIGIBLE):,} rows x {ELIGIBLE.shape[1]} cols")
assert len(ELIGIBLE) == 1883, f"unexpected eligible row count: {len(ELIGIBLE)}"
assert not (ELIGIBLE.pos.eq("QB") & ELIGIBLE.grp.eq("rook")).any()

_chk = ELIGIBLE.copy()
_chk["recon_model_pop"] = _chk.groupby(["season", "pos"])["adp_half_ppr"].rank(method="min")
SD = pd.read_csv(SEAS_CSV)
_full = SD[SD.adp_half_ppr.notna() & SD.season.isin(TEST_SEASONS)].copy()
_full["recon_full_dataset"] = _full.groupby(["season", "position"])["adp_half_ppr"].rank(method="min")

ADP_RANK_DIAG = {
    "stored_matches_recon_within_model_population": int((_chk.recon_model_pop == _chk.adp_pos_rank).sum()),
    "model_population_rows": int(len(_chk)),
    "stored_matches_recon_within_full_dataset": int((_full.recon_full_dataset == _full.adp_pos_rank).sum()),
    "full_dataset_adp_rows": int(len(_full)),
    "reason": ("adp_pos_rank is merged verbatim from an external ADP source CSV in "
               "build_season_dataset.py, ranked over that source's universe — a superset of both the "
               "season dataset and the model's scored population. Not reproducible from a subset; NOT used."),
}

print("\nSTORED adp_pos_rank vs RECONSTRUCTION")
print("-" * 92)
print(f"  matches within the model population : "
      f"{ADP_RANK_DIAG['stored_matches_recon_within_model_population']:,} / {len(_chk):,} "
      f"({ADP_RANK_DIAG['stored_matches_recon_within_model_population']/len(_chk):.1%})")
print(f"  matches within the full dataset     : "
      f"{ADP_RANK_DIAG['stored_matches_recon_within_full_dataset']:,} / {len(_full):,} "
      f"({ADP_RANK_DIAG['stored_matches_recon_within_full_dataset']/len(_full):.1%})")

print("\nWORKED EXAMPLE — 2025 WR by ADP, around the first divergence:")
_ex = (_full[(_full.season == 2025) & (_full.position == "WR")].sort_values("adp_half_ppr")
       [["player", "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "recon_full_dataset"]])
_i = _ex.index.get_loc(_ex.index[(_ex.adp_pos_rank != _ex.recon_full_dataset).to_numpy().argmax()])
print(_ex.iloc[max(0, _i - 3): _i + 4].to_string(index=False))
print("\n  ^ the stored rank jumps while the reconstruction does not: a player priced by the ADP")
print("    source sits in that gap but is absent from this dataset.")
print("\ngenerating line in build_season_dataset.py:")
for ln in (SEAS_DIR / "build_season_dataset.py").read_text(encoding="utf-8").splitlines():
    if "adp_pos_rank" in ln and "keep" in ln:
        print("   ", ln.strip())
(INTERIM / "stage02_adp_rank_diagnostic.json").write_text(json.dumps(ADP_RANK_DIAG, indent=2),
                                                          encoding="utf-8")
print("\n=> DECISION: rebuild every rank inside the stated population; never read adp_pos_rank.")

loaded interim/eligible_rows.csv: 1,883 rows x 16 cols

STORED adp_pos_rank vs RECONSTRUCTION
--------------------------------------------------------------------------------------------
  matches within the model population : 917 / 1,883 (48.7%)
  matches within the full dataset     : 981 / 1,915 (51.2%)

WORKED EXAMPLE — 2025 WR by ADP, around the first divergence:
         player  adp_half_ppr  adp_overall_rank  adp_pos_rank  recon_full_dataset
     Josh Downs         119.3             117.0          50.0                50.0
 Jayden Higgins         121.1             119.0          51.0                51.0
 Rashid Shaheed         125.9             123.0          52.0                52.0
 Darnell Mooney         132.0             129.0          54.0                53.0
   Keenan Allen         141.6             131.0          55.0                54.0
Marvin Mims Jr.         145.5             135.0          56.0                55.0
 Christian Kirk         148.5             137.0         

### Interpretation — the stored rank is not reproducible, and the worked example shows why

Neither reconstruction matches: **917 of 1,883 (48.7%)** within the model population, **981 of 1,915
(51.2%)** within the full ADP-bearing dataset. Widening the universe barely helps, which rules out
"we simply used too small a pool" as the explanation.

The worked example makes the mechanism concrete. Through 2025 WR, stored and reconstructed ranks agree
exactly — Josh Downs 50, Jayden Higgins 51, Rashid Shaheed 52 — and then **Darnell Mooney is stored as
54 while the reconstruction says 53**, with every later receiver staying one apart. A wide receiver
priced by the ADP source occupies slot 53 and is absent from this dataset entirely. The stored column
counts him; nothing available here can.

The grep confirms it at source: `adp_pos_rank` appears only in the `keep` list of a merge. It is
copied in, never computed here.

**The consequence is a genuine trade-off, taken deliberately.** Rebuilding ranks in-population makes
every number reproducible from the exported CSVs — stage 07 verifies exactly that on all 1,298 summary
cells — at the cost of not being the literal market-wide positional rank. It also means **rank values
depend on the population**, which is why the drafted board is a re-ranked population rather than a
filter over the full population's ranks, and why the two are never mixed in a single table.

### Explain — build both rank universes across both populations

`build_ranks()` from the shared library is applied across the 2x2 of universe x population, and the
result validated before any signal is derived.

**Universe A** ranks over the whole ADP-bearing population; a row with no Sleeper projection keeps a
missing Sleeper rank but still occupies the ADP, model and actual orderings — exactly as such a
player does on the real Draft Board, where he affects everyone else's rank. **Universe B** restricts
to rows complete on all four quantities first, then re-ranks inside that identical set.

**Population** is applied first via `population_slice`, so `drafted_top180` ranks are computed inside
the 180-player pool rather than inherited.

**Three validations, each catching a specific silent failure.**

1. **Universes A and B must agree exactly on which rows are complete**, checked by set comparison on
   `(season, player_id)`. Without this, any A-vs-B difference would be confounded by composition and
   the sensitivity would be uninterpretable — they must be the same players ranked differently, not
   different samples.
2. **Every rank must start at 1 and never exceed its season-position cell size**, across all 80
   groups. This is what catches a rank accidentally taken across positions, which would still produce
   a plausible-looking column.
3. Row and completeness counts are printed per combination so the Interpretation is grounded in
   observed state.

In [3]:
_stack, _rows = [], []
for pop_name, cap in POPULATIONS.items():
    base = population_slice(ELIGIBLE, cap)
    for uni in ("A", "B"):
        d = build_ranks(base, uni)
        d["population"] = pop_name
        _stack.append(d)
        _rows.append({"population": pop_name, "universe": uni, "rows": len(d),
                      "complete": int(d.complete.sum()), "incomplete": int((~d.complete).sum()),
                      "season_pos_cells": int(d.groupby(["season", "pos"]).ngroups)})
RANKED = pd.concat(_stack, ignore_index=True)

print("RANK CONSTRUCTION — 2 universes x 2 populations")
print(pd.DataFrame(_rows).to_string(index=False))

print("\nVALIDATION 1 — A and B must agree on the COMPLETE row set within a population:")
for pop_name in POPULATIONS:
    a = RANKED[(RANKED.population == pop_name) & (RANKED.universe == "A") & RANKED.complete]
    b = RANKED[(RANKED.population == pop_name) & (RANKED.universe == "B") & RANKED.complete]
    same = set(zip(a.season, a.player_id)) == set(zip(b.season, b.player_id))
    print(f"  {pop_name:15s} identical membership: {same}  (A={len(a):,}, B={len(b):,})")
    assert same, "universes disagree on which rows are complete"

print("\nVALIDATION 2 — ranks must run 1..cell_size within every (population, universe, season, pos):")
_bad = []
for keys, g in RANKED.groupby(["population", "universe", "season", "pos"]):
    for col in ("adp_rank", "model_rank", "actual_rank"):
        if g[col].min() != 1 or g[col].max() > len(g):
            _bad.append((keys, col, float(g[col].min()), float(g[col].max()), len(g)))
print(f"  season-position groups checked : {RANKED.groupby(['population','universe','season','pos']).ngroups}")
print(f"  malformed rank cells           : {len(_bad)}")
assert not _bad, f"rank construction produced out-of-range ranks: {_bad[:5]}"
print(f"\n  example cell sizes (all_adp / A): "
      f"{dict(list(RANKED[(RANKED.population=='all_adp')&(RANKED.universe=='A')].groupby(['season','pos']).size().items())[:4])}")

RANK CONSTRUCTION — 2 universes x 2 populations
    population universe  rows  complete  incomplete  season_pos_cells
       all_adp        A  1883      1679         204                20
       all_adp        B  1679      1679           0                20
drafted_top180        A   886       867          19                20
drafted_top180        B   867       867           0                20

VALIDATION 1 — A and B must agree on the COMPLETE row set within a population:
  all_adp         identical membership: True  (A=1,679, B=1,679)
  drafted_top180  identical membership: True  (A=867, B=867)

VALIDATION 2 — ranks must run 1..cell_size within every (population, universe, season, pos):
  season-position groups checked : 80
  malformed rank cells           : 0

  example cell sizes (all_adp / A): {(2021, 'QB'): 50, (2021, 'RB'): 97, (2021, 'TE'): 44, (2021, 'WR'): 112}


### Interpretation — ranks are well-formed and the two universes are consistent

The 2x2 built cleanly. `all_adp` universe A holds 1,883 rows of which **1,679 are complete** (204 lack
a Sleeper rank); universe B holds exactly those 1,679. `drafted_top180` A holds 886 rows with **867
complete** (19 incomplete); B holds 867. All four combinations span the same **20 season-position
cells** (5 seasons x 4 positions), so none is silently missing.

**Validation 1 passed in both populations**: A and B contain identical complete-row membership. That
is the guarantee that makes the A-vs-B comparison in stage 03 a genuine sensitivity — the same players
ranked differently, not two different samples whose difference could be composition.

**Validation 2 passed on all 80 groups**: zero malformed cells. Every rank starts at 1 and none
exceeds its cell size. This is the check that catches a `groupby` key quietly dropping — ranking
across positions would produce maxima far above the cell size while still looking like a rank column.
Stage 07 re-derives these ranks from scratch by a separate expression and confirms the same property.

One asymmetry worth noting: incompleteness is much lower on the drafted board (**19 of 886, 2.1%**)
than on the full population (**204 of 1,883, 10.8%**). Sleeper covers drafted players far better than
deep ones — itself a hint that the deep tail is a different kind of data, which stage 03 pursues.

### Explain — derive the signal and write the row-level analysis frame

`add_signals()` from the shared library turns ranks into the quantity being tested, and the result is
exported as `artifacts/player_season_results.csv`.

**What it computes:** the three signed gaps (positive = ranked above his draft price), the
same-direction agreement flag, the per-threshold eligibility booleans, the consensus score carrying
the shared sign on the **weaker** gap, the buy/fade direction, and the hit/miss/tie outcome with an
exact tie scored a **miss**.

**A worked example** is printed for a single real player, showing the arithmetic end to end — ADP to
rank, both projections to ranks and gaps, actual to gap, and the resulting consensus and outcome. One
concrete row makes the definitions checkable in a way a formula does not.

**The eligible-cell table** is the number that shapes every later stage: how many agreement calls each
population and threshold actually produces.

**Export.** 5,315 rows — the four population x universe combinations stacked, each row carrying its
own correctly-scoped ranks. A reader must filter on `population` and `universe` before aggregating;
the row count is deliberately not a player count.

In [4]:
SIGNALS = add_signals(RANKED)

print("SIGNAL DEFINITIONS APPLIED")
print("-" * 96)
print("  model_gap   = adp_rank - model_rank      (positive: we rank him ABOVE his draft price)")
print("  sleeper_gap = adp_rank - sleeper_rank")
print("  actual_gap  = adp_rank - actual_rank")
print("  consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)   [weaker gap governs]")
print("  hit = sign(actual_gap) == sign(consensus_score) AND actual_gap != 0;  tie scores as a MISS")

_ex = (SIGNALS[(SIGNALS.population == "drafted_top180") & (SIGNALS.universe == "A")
               & (SIGNALS.season == 2024) & SIGNALS[thr_col(10.0)]].nlargest(1, "consensus_score").iloc[0])
print(f"\nWORKED EXAMPLE — {_ex.player} ({_ex.pos}, {int(_ex.season)}), drafted_top180 / universe A")
print(f"  ADP overall {_ex.adp_half_ppr:>6.1f}  ->  adp_rank     = {int(_ex.adp_rank)}")
print(f"  model pred  {_ex.pred:>6.1f}  ->  model_rank   = {int(_ex.model_rank)}   model_gap   = {int(_ex.model_gap):+d}")
print(f"  sleeper     {_ex.sleeper:>6.1f}  ->  sleeper_rank = {int(_ex.sleeper_rank)}   sleeper_gap = {int(_ex.sleeper_gap):+d}")
print(f"  actual      {_ex.y:>6.1f}  ->  actual_rank  = {int(_ex.actual_rank)}   actual_gap  = {int(_ex.actual_gap):+d}")
print(f"  agree={_ex.agree_dir}  consensus={int(_ex.consensus_score):+d} ({_ex.direction})  -> {_ex.outcome.upper()}")

print("\nELIGIBLE AGREEMENT CELLS (all seasons 2021-2025):")
_tbl = []
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        d = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni) & SIGNALS.complete]
        _tbl.append({"population": pop_name, "universe": uni, "complete_rows": len(d),
                     **{f"t>{t:g}": int(d[thr_col(t)].sum()) for t in THRESHOLDS}})
print(pd.DataFrame(_tbl).to_string(index=False))

print("\nDirection split at t>0 (complete rows):")
print(SIGNALS[(SIGNALS.universe == "A") & SIGNALS.complete]
      .groupby(["population", "direction"]).size().unstack(fill_value=0).to_string())

EXPORT_COLS = ["population", "universe", "season", "pos", "player_id", "player", "team", "group",
               "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "pred", "sleeper", "y",
               "adp_rank", "model_rank", "sleeper_rank", "actual_rank",
               "model_gap", "sleeper_gap", "actual_gap", "agree_dir", "consensus_score", "direction",
               "complete"] + [thr_col(t) for t in THRESHOLDS] + ["outcome", "model"]
PLAYER_RESULTS = (SIGNALS[EXPORT_COLS].rename(columns={"pos": "position", "pred": "model_pred",
                                                       "y": "actual_half_ppr", "model": "model_family"})
                  .sort_values(["population", "universe", "season", "position", "adp_rank"])
                  .reset_index(drop=True))
PLAYER_RESULTS["max_elig_threshold"] = np.select(
    [PLAYER_RESULTS[thr_col(t)] for t in sorted(THRESHOLDS, reverse=True)],
    sorted(THRESHOLDS, reverse=True), default=np.nan)
PLAYER_RESULTS.to_csv(ARTIFACTS / "player_season_results.csv", index=False)
print(f"\nwrote artifacts/player_season_results.csv — {len(PLAYER_RESULTS):,} rows x "
      f"{PLAYER_RESULTS.shape[1]} cols")

SIGNAL DEFINITIONS APPLIED
------------------------------------------------------------------------------------------------
  model_gap   = adp_rank - model_rank      (positive: we rank him ABOVE his draft price)
  sleeper_gap = adp_rank - sleeper_rank
  actual_gap  = adp_rank - actual_rank
  consensus_score = sign(model_gap) * min(|model_gap|, |sleeper_gap|)   [weaker gap governs]
  hit = sign(actual_gap) == sign(consensus_score) AND actual_gap != 0;  tie scores as a MISS

WORKED EXAMPLE — Michael Wilson (WR, 2024), drafted_top180 / universe A
  ADP overall  201.7  ->  adp_rank     = 73
  model pred   100.4  ->  model_rank   = 55   model_gap   = +18
  sleeper      116.9  ->  sleeper_rank = 58   sleeper_gap = +15
  actual       101.0  ->  actual_rank  = 49   actual_gap  = +24
  agree=True  consensus=+15 (buy)  -> HIT

ELIGIBLE AGREEMENT CELLS (all seasons 2021-2025):
    population universe  complete_rows  t>0  t>5  t>7.5  t>10
       all_adp        A           1679  940  516    422   


wrote artifacts/player_season_results.csv — 5,315 rows x 32 cols


### Interpretation — the arithmetic on one player, and the cells it produces

The worked example is the whole study in six lines. **Michael Wilson, WR, 2024**: an ADP of pick 201.7
makes him WR73; the model projects him WR55 (`model_gap = +18`); Sleeper projects him WR58
(`sleeper_gap = +15`). Both say underpriced, so `agree_dir` is true and `consensus_score = +15` — the
**weaker** of the two, Sleeper's, not the model's larger +18. He finished WR49, `actual_gap = +24`,
and the call is a hit.

That row also shows why the minimum is the right choice: had the model said +18 and Sleeper +2, the
pair would score +2 and fall out of every threshold above 0. Neither source can carry an agreement
alone.

The eligible-cell table sets expectations for everything downstream, and the two populations behave
very differently:

- `all_adp` universe A: **940 / 516 / 422 / 331** calls at t>0 / >5 / >7.5 / >10 — the strictest
  threshold still keeps 331 calls, **35%** of the t>0 cell.
- `drafted_top180` universe A: **410 / 105 / 70 / 37** — the strictest keeps only **9%**, about seven
  calls per season across five seasons.

That contrast is itself a warning. Large rank gaps are far more *available* when the rank space runs
215 receivers deep than when it stops at the draftable pool. Whether the extra calls are signal or
noise is exactly what stage 03 asks.

Direction is close to balanced (`all_adp` 805 buy / 701 fade; drafted 380 / 351), so no result
downstream is a one-sided artifact of both systems being uniformly more optimistic than the market.

# Conclusion and Next Steps

## What this stage established

**The stored `adp_pos_rank` is unusable and is not used.** It matches an in-population reconstruction
on only **48.7%** of rows (917 of 1,883) and an in-dataset reconstruction on **51.2%**, because it is
merged verbatim from an external ADP source ranked over a larger universe. The worked example makes
the mechanism visible: the stored rank skips a slot occupied by a player the source priced and this
dataset never retained. Every rank in this study is rebuilt in-population instead — reproducible, and
population-dependent by construction.

**Ranks are well-formed.** Zero malformed season-position cells across all 80 groups; universes A and
B agree exactly on which rows are complete, so they are the same players ranked differently rather
than different samples.

**The signal is built.** Eligible agreement cells, universe A: `all_adp` **940 / 516 / 422 / 331** and
`drafted_top180` **410 / 105 / 70 / 37** at t>0 / >5 / >7.5 / >10.

## What is now true

`artifacts/player_season_results.csv` holds 5,315 rows — the four population x universe combinations,
each row carrying its own correctly-scoped ranks, gaps, eligibility flags and outcome. Stage 07
re-derives every rank in that file from scratch and confirms it.

## The number to carry forward

The drafted board keeps only **9%** of its threshold-0 cell at `t>10` (37 of 410), against **35%** for
the full population (331 of 940). Large rank gaps are far more *available* when the rank space runs
215 receivers deep than when it stops at the draftable pool. Whether those extra calls are signal or
noise is exactly what stage 03 asks.

## Next step

Run **`03_main_results.ipynb`** for the primary tables and the artifact test.